# 02. 속도와 품질을 함께 측정하기

목표: AI 도입 전후의 흐름을 비교하되 처리량만 보지 않고 리드타임, 첫 CI 성공률, 재작업률, 변경 실패율을 함께 평가합니다. 숫자는 학습용 가상 데이터입니다.

In [ ]:
from dataclasses import dataclass
from statistics import mean

@dataclass(frozen=True)
class Change:
    lead_hours: float
    first_ci_pass: bool
    rework_cycles: int
    deployed: bool
    failed: bool

before = [
    Change(96, True, 2, True, False), Change(120, False, 4, True, True),
    Change(80, True, 2, True, False), Change(144, False, 5, True, False),
    Change(72, True, 1, True, False),
]
after = [
    Change(32, True, 1, True, False), Change(28, True, 1, True, False),
    Change(24, False, 3, True, True), Change(36, True, 1, True, False),
    Change(20, True, 0, True, False), Change(30, True, 1, True, False),
]


In [ ]:
def metrics(changes):
    deployed = [item for item in changes if item.deployed]
    return {
        '평균 리드타임(시간)': mean(item.lead_hours for item in changes),
        '첫 CI 성공률': mean(item.first_ci_pass for item in changes),
        '평균 재작업 횟수': mean(item.rework_cycles for item in changes),
        '변경 실패율': mean(item.failed for item in deployed),
        '표본 수': len(changes),
    }

for label, dataset in [('도입 전', before), ('도입 후', after)]:
    print(label)
    for key, value in metrics(dataset).items():
        print(f'  {key}: {value:.3f}' if isinstance(value, float) else f'  {key}: {value}')


## 가드레일이 있는 확대 결정

속도 개선만으로 자동화 범위를 넓히지 않습니다. 품질 지표가 허용 범위를 지키고 표본이 충분해야 다음 단계로 갑니다.

In [ ]:
def adoption_gate(baseline, candidate, minimum_sample=5):
    old, new = metrics(baseline), metrics(candidate)
    checks = {
        '표본 충분': new['표본 수'] >= minimum_sample,
        '리드타임 20% 이상 감소': new['평균 리드타임(시간)'] <= old['평균 리드타임(시간)'] * 0.8,
        '첫 CI 성공률 비열화': new['첫 CI 성공률'] >= old['첫 CI 성공률'],
        '변경 실패율 비열화': new['변경 실패율'] <= old['변경 실패율'],
        '재작업 감소': new['평균 재작업 횟수'] < old['평균 재작업 횟수'],
    }
    return all(checks.values()), checks

approved, checks = adoption_gate(before, after)
print('자동화 확대:', approved)
for name, passed in checks.items():
    print('PASS' if passed else 'FAIL', name)
assert approved


## 확장 과제

표본 신뢰구간, 변경 위험도, 작업 크기를 추가하세요. 고위험 변경과 저위험 변경을 섞은 평균이 중요한 실패를 숨기지 않는지도 확인해 보세요.